# AlphaZero-like Chess Engine

This notebook implements a neural network for playing chess, inspired by AlphaZero. It learns through a combination of self-play and playing against Stockfish. The implementation includes features like GPU acceleration, resumable training, and a comprehensive evaluation script.

## 1. Setup and Installation

First, we need to install the necessary libraries. `torch` for the neural network, `python-chess` for chess logic and engine communication, and `stockfish` for the Stockfish engine wrapper. `tqdm` is used for progress bars.

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install python-chess --quiet
!pip install stockfish --quiet
!pip install tqdm --quiet

## 2. Imports and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import chess
import numpy as np
from stockfish import Stockfish
import random
import os
import pickle
from collections import deque
import time
import math
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning) # To suppress some tqdm warnings in notebooks

# Configuration
class Config:
    def __init__(self):
        # --- General ---
        self.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.DEVICE}")
        self.CHECKPOINT_DIR = "checkpoints"
        self.REPLAY_BUFFER_PATH = "replay_buffer.pkl"

        # --- Neural Network ---
        self.NN_INPUT_CHANNELS = 18 # From board_to_tensor
        self.NUM_RES_BLOCKS = 19
        self.NUM_FILTERS = 256
        self.POLICY_OUTPUT_SIZE = 4672 # Placeholder, will be updated by MoveConverter

        # --- MCTS ---
        self.NUM_SIMULATIONS = 100 # Reduced for faster iteration, increase for strength (e.g., 800)
        self.CPUCT = 1.25
        self.TEMPERATURE_START = 1.0
        self.TEMPERATURE_END = 0.1
        self.TEMPERATURE_DECAY = 30
        self.DIRICHLET_ALPHA = 0.3
        self.DIRICHLET_EPSILON = 0.25
        
        # --- Training ---
        self.NUM_ITERATIONS = 1000
        self.SELF_PLAY_GAMES_PER_ITER = 20
        self.STOCKFISH_GAMES_PER_ITER = 5
        self.REPLAY_BUFFER_SIZE = 50000
        self.BATCH_SIZE = 2048
        self.LEARNING_RATE = 1e-3
        self.WEIGHT_DECAY = 1e-4
        self.CHECKPOINT_INTERVAL = 5

        # --- Stockfish ---
        # Download stockfish from: https://stockfishchess.org/download/
        # And place the executable in the same directory or provide full path
        try:
            self.STOCKFISH_PATH = Stockfish()._stockfish_path
        except Exception as e:
            print(f"Could not auto-detect Stockfish path: {e}")
            self.STOCKFISH_PATH = "stockfish" # UPDATE THIS PATH if needed
        self.STOCKFISH_SKILL_LEVEL_TRAINING = 20

        # --- Evaluation ---
        self.EVAL_GAMES = 10
        self.EVAL_STOCKFISH_SKILL_LEVELS = list(range(1, 21, 2))

config = Config()

os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

## 3. Move Encoding and Decoding

To interface with the neural network, we need a fixed-size representation for chess moves. We use a vector space large enough to encode all possible moves from any square to any other square, including promotions. A pre-computed mapping from UCI strings to integer actions is used for simplicity and robustness.

In [ ]:
class MoveConverter:
    def __init__(self):
        self.uci_to_action = {}
        self.action_to_uci = []
        self._create_move_mappings()

    def _create_move_mappings(self):
        # Create a comprehensive list of all possible moves in UCI format.
        # This is a simplified but effective approach.
        squares = range(64)
        for from_sq in squares:
            for to_sq in squares:
                if from_sq == to_sq: continue
                # Normal moves
                self._add_move(chess.Move(from_sq, to_sq).uci())
                # Promotion moves
                from_rank = chess.square_rank(from_sq)
                to_rank = chess.square_rank(to_sq)
                if (from_rank == 6 and to_rank == 7) or (from_rank == 1 and to_rank == 0):
                    for promo_piece in [chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN]:
                        self._add_move(chess.Move(from_sq, to_sq, promotion=promo_piece).uci())

    def _add_move(self, uci):
        if uci not in self.uci_to_action:
            action = len(self.action_to_uci)
            self.uci_to_action[uci] = action
            self.action_to_uci.append(uci)

    def move_to_action(self, move: chess.Move):
        return self.uci_to_action.get(move.uci())

    def action_to_move(self, action: int, board: chess.Board):
        uci = self.action_to_uci[action]
        try:
            # The move must be legal in the current position to be valid
            move = board.parse_uci(uci)
            if move in board.legal_moves:
                 return move
        except (chess.InvalidMoveError, chess.IllegalMoveError):
            pass
        return None

move_converter = MoveConverter()
config.POLICY_OUTPUT_SIZE = len(move_converter.action_to_uci)
print(f"Policy output size set to: {config.POLICY_OUTPUT_SIZE}")

## 4. Chess Representation

The chess board is converted into a multi-channel tensor. This representation is inspired by AlphaZero and captures all necessary information for the neural network.
We use an (18, 8, 8) tensor:
- 12 planes for piece positions (6 for white, 6 for black)
- 1 plane for active color
- 1 plane for total move count
- 4 planes for castling rights

In [ ]:
def board_to_tensor(board: chess.Board):
    tensor = np.zeros((config.NN_INPUT_CHANNELS, 8, 8), dtype=np.float32)
    piece_map = {chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5}

    for i in range(64):
        piece = board.piece_at(i)
        if piece:
            color_offset = 0 if piece.color == board.turn else 6
            piece_idx = piece_map[piece.piece_type]
            row, col = i // 8, i % 8
            tensor[piece_idx + color_offset, row, col] = 1

    tensor[12, :, :] = 1 # Player to move is always the 'first' perspective
    tensor[13, :, :] = board.fullmove_number / 100.0
    tensor[14, :, :] = 1 if board.has_kingside_castling_rights(board.turn) else 0
    tensor[15, :, :] = 1 if board.has_queenside_castling_rights(board.turn) else 0
    tensor[16, :, :] = 1 if board.has_kingside_castling_rights(not board.turn) else 0
    tensor[17, :, :] = 1 if board.has_queenside_castling_rights(not board.turn) else 0

    return torch.from_numpy(tensor)

## 5. Neural Network

This is the core of the agent. It's a ResNet-based architecture with two heads:
- **Policy Head:** Outputs a probability distribution over all possible moves.
- **Value Head:** Outputs a single value estimating the game's outcome from the current position.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, num_filters):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_filters)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = F.relu(out)
        return out

class AlphaZeroNet(nn.Module):
    def __init__(self):
        super(AlphaZeroNet, self).__init__()
        self.conv_in = nn.Conv2d(config.NN_INPUT_CHANNELS, config.NUM_FILTERS, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn_in = nn.BatchNorm2d(config.NUM_FILTERS)
        self.res_blocks = nn.ModuleList([ResBlock(config.NUM_FILTERS) for _ in range(config.NUM_RES_BLOCKS)])
        
        self.policy_conv = nn.Conv2d(config.NUM_FILTERS, 2, kernel_size=1, stride=1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * 8 * 8, config.POLICY_OUTPUT_SIZE)
        
        self.value_conv = nn.Conv2d(config.NUM_FILTERS, 1, kernel_size=1, stride=1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(1 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.bn_in(self.conv_in(x)))
        for block in self.res_blocks:
            x = block(x)
        
        policy = F.relu(self.policy_bn(self.policy_conv(x)))
        policy = policy.view(-1, 2 * 8 * 8)
        policy = self.policy_fc(policy)
        
        value = F.relu(self.value_bn(self.value_conv(x)))
        value = value.view(-1, 1 * 8 * 8)
        value = F.relu(self.value_fc1(value))
        value = torch.tanh(self.value_fc2(value))
        
        return F.log_softmax(policy, dim=1), value

## 6. Monte Carlo Tree Search (MCTS)

MCTS is a search algorithm that explores the game tree. It uses the neural network's outputs (policy and value) to guide its search and find the best move. The core idea is to use the PUCT algorithm to balance exploration and exploitation.

In [ ]:
class Node:
    def __init__(self, parent=None, prior_p=1.0):
        self.parent = parent
        self.children = {}
        self.n_visits = 0
        self.q_value = 0.0
        self.prior_p = prior_p

    def expand(self, policy, legal_moves):
        for move in legal_moves:
            action = move_converter.move_to_action(move)
            if action is not None and action not in self.children:
                self.children[action] = Node(parent=self, prior_p=policy[action])

    def select_child(self):
        best_score = -float('inf')
        best_action = -1
        best_child = None
        for action, child in self.children.items():
            score = child.get_value()
            if score > best_score:
                best_score = score
                best_action = action
                best_child = child
        return best_action, best_child

    def get_value(self):
        u_value = (config.CPUCT * self.prior_p * math.sqrt(self.parent.n_visits) / (1 + self.n_visits))
        return self.q_value + u_value

    def update_recursive(self, value):
        if self.parent:
            self.parent.update_recursive(-value)
        self.n_visits += 1
        self.q_value += (value - self.q_value) / self.n_visits

def mcts(board, model, add_exploration_noise=True):
    root = Node()
    for _ in range(config.NUM_SIMULATIONS):
        node = root
        sim_board = board.copy()
        
        while node.children:
            action, node = node.select_child()
            move = move_converter.action_to_move(action, sim_board)
            if move is None: continue
            sim_board.push(move)
        
        if not sim_board.is_game_over(claim_draw=True):
            board_tensor = board_to_tensor(sim_board).unsqueeze(0).to(config.DEVICE)
            with torch.no_grad():
                log_policy, value_tensor = model(board_tensor)
            policy = torch.exp(log_policy).squeeze(0).cpu().numpy()
            value = value_tensor.item()

            legal_moves = list(sim_board.legal_moves)
            if add_exploration_noise:
                legal_actions = [move_converter.move_to_action(m) for m in legal_moves if m is not None]
                noise = np.random.dirichlet([config.DIRICHLET_ALPHA] * len(legal_actions))
                policy_noisy = np.zeros_like(policy)
                for i, action in enumerate(legal_actions):
                    if action is not None:
                       policy_noisy[action] = (1 - config.DIRICHLET_EPSILON) * policy[action] + config.DIRICHLET_EPSILON * noise[i]
                policy = policy_noisy

            node.expand(policy, legal_moves)
        else:
            result = sim_board.result(claim_draw=True)
            if result == '1-0': value = 1.0
            elif result == '0-1': value = -1.0
            else: value = 0.0
        
        node.update_recursive(-value)
    
    return {action: child.n_visits for action, child in root.children.items()}

## 7. Data Generation

We generate training data from two sources:
1.  **Self-Play:** The agent plays against itself. This is the primary way it discovers new strategies.
2.  **vs. Stockfish:** The agent plays against a strong Stockfish engine to learn from an expert.

In [ ]:
def get_move_and_pi(board, model, temp, move_count):
    move_visits = mcts(board, model)
    if not move_visits: return None, None

    pi = np.zeros(config.POLICY_OUTPUT_SIZE, dtype=np.float32)
    total_visits = sum(move_visits.values())
    for action, visits in move_visits.items():
        pi[action] = visits / total_visits
    
    actions = list(move_visits.keys())
    visit_counts = np.array(list(move_visits.values()))
    
    current_temp = temp if move_count < config.TEMPERATURE_DECAY else 0
    if current_temp == 0:
        action = actions[np.argmax(visit_counts)]
    else:
        powered_counts = visit_counts ** (1 / current_temp)
        powered_counts /= np.sum(powered_counts)
        action = np.random.choice(actions, p=powered_counts)
        
    move = move_converter.action_to_move(action, board)
    return move, pi

def play_game(model, stockfish_instance=None, model_plays_white=True):
    board = chess.Board()
    game_history = []
    move_count = 0

    while not board.is_game_over(claim_draw=True):
        is_model_turn = (stockfish_instance is None) or \
                        (board.turn == chess.WHITE and model_plays_white) or \
                        (board.turn == chess.BLACK and not model_plays_white)

        if is_model_turn:
            move, pi = get_move_and_pi(board, model, config.TEMPERATURE_START, move_count)
            if move is None: break
            game_history.append([board_to_tensor(board), pi, 0])
            board.push(move)
        else:
            stockfish_instance.set_fen_position(board.fen())
            move_uci = stockfish_instance.get_best_move()
            if move_uci is None: break
            board.push(chess.Move.from_uci(move_uci))
        
        move_count += 1

    result = board.result(claim_draw=True)
    if result == '1-0': final_reward = 1
    elif result == '0-1': final_reward = -1
    else: final_reward = 0

    # Assign rewards from the perspective of the player who made the move
    for i, data in enumerate(game_history):
        if stockfish_instance is None: # Self-play
            is_white_move = (i % 2 == 0)
            data[2] = final_reward if is_white_move else -final_reward
        else: # vs. Stockfish
            data[2] = final_reward if model_plays_white else -final_reward
            
    return game_history


## 8. Training Loop

The main loop orchestrates the learning process:
1.  **Generate Data:** Play games (self-play and vs. Stockfish) to collect new training examples.
2.  **Store Data:** Add the new data to a replay buffer.
3.  **Train Model:** Sample a batch from the replay buffer and train the neural network.
4.  **Save Checkpoint:** Periodically save the model's state.

In [ ]:
def save_checkpoint(model, optimizer, iteration, replay_buffer):
    filepath = os.path.join(config.CHECKPOINT_DIR, f"checkpoint_{iteration}.pth")
    state = {
        'iteration': iteration,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }
    torch.save(state, filepath)
    
    buffer_path = os.path.join(config.CHECKPOINT_DIR, config.REPLAY_BUFFER_PATH)
    with open(buffer_path, 'wb') as f:
        pickle.dump(replay_buffer, f)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(model, optimizer):
    checkpoints = [f for f in os.listdir(config.CHECKPOINT_DIR) if f.endswith('.pth')]
    if not checkpoints:
        print("No checkpoint found, starting from scratch.")
        return 0, deque(maxlen=config.REPLAY_BUFFER_SIZE)
    
    latest_checkpoint_name = max(checkpoints, key=lambda f: int(f.split('_')[1].split('.')[0]))
    filepath = os.path.join(config.CHECKPOINT_DIR, latest_checkpoint_name)
    
    checkpoint = torch.load(filepath, map_location=config.DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_iter = checkpoint['iteration'] + 1
    
    replay_buffer = deque(maxlen=config.REPLAY_BUFFER_SIZE)
    buffer_path = os.path.join(config.CHECKPOINT_DIR, config.REPLAY_BUFFER_PATH)
    if os.path.exists(buffer_path):
        with open(buffer_path, 'rb') as f:
            replay_buffer = pickle.load(f)
    
    print(f"Loaded checkpoint from {filepath}, resuming from iteration {start_iter}")
    return start_iter, replay_buffer

def train_step(model, optimizer, replay_buffer):
    if len(replay_buffer) < config.BATCH_SIZE:
        return 0, 0
    
    model.train()
    samples = random.sample(replay_buffer, config.BATCH_SIZE)
    boards, pis, vs = zip(*samples)
    
    boards = torch.stack(boards).to(config.DEVICE)
    pis = torch.from_numpy(np.array(pis)).to(config.DEVICE)
    vs = torch.tensor(vs, dtype=torch.float32).unsqueeze(1).to(config.DEVICE)
    
    optimizer.zero_grad()
    log_policy_preds, value_preds = model(boards)
    
    policy_loss = F.kl_div(log_policy_preds, pis, reduction='batchmean')
    value_loss = F.mse_loss(value_preds, vs)
    loss = policy_loss + value_loss
    
    loss.backward()
    optimizer.step()
    
    return policy_loss.item(), value_loss.item()

def run_training_loop(resume=False):
    model = AlphaZeroNet().to(config.DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    
    start_iter = 0
    replay_buffer = deque(maxlen=config.REPLAY_BUFFER_SIZE)
    if resume:
        start_iter, replay_buffer = load_checkpoint(model, optimizer)

    stockfish = Stockfish(path=config.STOCKFISH_PATH, parameters={"Skill Level": config.STOCKFISH_SKILL_LEVEL_TRAINING})

    for i in range(start_iter, config.NUM_ITERATIONS):
        print(f"\n--- Iteration {i+1}/{config.NUM_ITERATIONS} ---")
        model.eval()
        
        with tqdm(total=config.SELF_PLAY_GAMES_PER_ITER, desc="Self-Play") as pbar:
            for _ in range(config.SELF_PLAY_GAMES_PER_ITER):
                game_data = play_game(model)
                replay_buffer.extend(game_data)
                pbar.update(1)
        
        with tqdm(total=config.STOCKFISH_GAMES_PER_ITER, desc="vs. Stockfish") as pbar:
            for _ in range(config.STOCKFISH_GAMES_PER_ITER):
                model_plays_white = random.choice([True, False])
                game_data = play_game(model, stockfish_instance=stockfish, model_plays_white=model_plays_white)
                replay_buffer.extend(game_data)
                pbar.update(1)

        print(f"Training on {len(replay_buffer)} samples...")
        policy_loss, value_loss = train_step(model, optimizer, replay_buffer)
        print(f"Policy Loss: {policy_loss:.4f}, Value Loss: {value_loss:.4f}")

        if (i + 1) % config.CHECKPOINT_INTERVAL == 0:
            save_checkpoint(model, optimizer, i, replay_buffer)


## 9. Evaluation

To gauge the model's strength, we play it against Stockfish at various skill levels. The evaluation stops when the model loses a match.

In [ ]:
def evaluate(model_path, stockfish_path):
    model = AlphaZeroNet().to(config.DEVICE)
    try:
        checkpoint = torch.load(model_path, map_location=config.DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
    except Exception as e:
        print(f"Could not load model from {model_path}: {e}")
        return
    model.eval()
    
    try:
        stockfish = Stockfish(path=stockfish_path)
    except Exception as e:
        print(f"Could not initialize Stockfish from {stockfish_path}: {e}")
        return
    
    for skill in config.EVAL_STOCKFISH_SKILL_LEVELS:
        stockfish.set_skill_level(skill)
        print(f"\n--- Evaluating against Stockfish skill level: {skill} ---")
        wins, losses, draws = 0, 0, 0
        
        for i in tqdm(range(config.EVAL_GAMES), desc=f"Skill {skill}"):
            board = chess.Board()
            is_model_white = i % 2 == 0
            
            while not board.is_game_over(claim_draw=True):
                if (board.turn == chess.WHITE and is_model_white) or \
                   (board.turn == chess.BLACK and not is_model_white):
                    move, _ = get_move_and_pi(board, model, temp=0, move_count=999) # Greedy move
                    if move is None: break
                    board.push(move)
                else:
                    stockfish.set_fen_position(board.fen())
                    best_move_uci = stockfish.get_best_move()
                    if best_move_uci is None: break
                    board.push(chess.Move.from_uci(best_move_uci))
            
            result = board.result(claim_draw=True)
            if result == '1-0':
                if is_model_white: wins += 1
                else: losses += 1
            elif result == '0-1':
                if not is_model_white: wins += 1
                else: losses += 1
            else:
                draws += 1
        
        print(f"Results vs Skill {skill}: {wins} Wins, {losses} Losses, {draws} Draws")
        if losses > wins:
            print("Model lost more games than it won. Stopping evaluation.")
            break

## 10. Main Execution

Use the cells below to start or resume training, or to evaluate a trained model. Make sure to uncomment the line for the function you want to run.

In [ ]:
def main():
    # --- To start training from scratch ---
    # print("Starting training from scratch...")
    # run_training_loop(resume=False)

    # --- To resume training from the latest checkpoint ---
    # print("Resuming training from latest checkpoint...")
    # run_training_loop(resume=True)
    
    # --- To run evaluation on a saved model ---
    # LATEST_CHECKPOINT = 'checkpoints/checkpoint_9.pth' # <--- UPDATE THIS PATH
    # if os.path.exists(LATEST_CHECKPOINT):
    #     print(f"Running evaluation on {LATEST_CHECKPOINT}...")
    #     evaluate(LATEST_CHECKPOINT, config.STOCKFISH_PATH)
    # else:
    #     print(f"Checkpoint {LATEST_CHECKPOINT} not found. Cannot run evaluation.")
    
    print("Notebook setup is complete. To begin, edit the `main` function and uncomment the desired action.")
    print(f"Your Stockfish path is configured to: '{config.STOCKFISH_PATH}'")
    print("Before running, ensure this path is correct or update it in the Config class.")

# This check allows the notebook to be imported as a module if needed.
if __name__ == '__main__':
    main()